In [2]:
from Scene_lib2 import *
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler
import pickle

In [3]:
import os
import numpy as np

ns = np.arange(start=50, stop=1501, step=50)
categories = ['Measured']
noise_levels = [0.]
base_dir = '3D-Data'
h = 3.5
boundary = [[-4,4],[-4,4],[0,h]]

# Generate 10 datasets with different seeds
num_seeds = 10

for category in categories:
    for noise_level in noise_levels:
        for n in ns:
            for seed in range(num_seeds):  # Iterate over 10 seeds (0 to 9)
                my_scene = Scene()
                use_measured_data = (category == 'Measured')

                my_scene.generate_sven_lights(use_measured_data=use_measured_data)
                my_scene.generate_diode_plane(n=2)
                my_scene.generate_diode_plane(n=2, h=h)
                my_scene.generate_diode_halton_volume(n=n-8, volume_boundary=[[-4,4],[-4,4],[0,h]], seed=seed)

                # Generate RSS with distances
                data = my_scene.generate_rss_table_with_diode_coords_and_distances(noise_std=noise_level, seed=seed)

                # Drop "X", "Y", "Z" columns to keep only log_d relevant data
                data = data.drop(columns=["X", "Y", "Z"])

                # Save into the "Measured_D" directory (for log_d data)
                category_dir = os.path.join(base_dir, "Measured_D")
                os.makedirs(category_dir, exist_ok=True)

                # Save the dataset with seed info in filename
                file_name = f"rss_n={n}_noise={noise_level}_seed={seed}.csv"
                file_path = os.path.join(category_dir, file_name)

                data.to_csv(file_path, index=False)
                print(f"Saved log_d data for {category} (n={n}, noise={noise_level}, seed={seed}) to {file_path}")


Saved log_d data for Measured (n=50, noise=0.0, seed=0) to 3D-Data\Measured_D\rss_n=50_noise=0.0_seed=0.csv
Saved log_d data for Measured (n=50, noise=0.0, seed=1) to 3D-Data\Measured_D\rss_n=50_noise=0.0_seed=1.csv
Saved log_d data for Measured (n=50, noise=0.0, seed=2) to 3D-Data\Measured_D\rss_n=50_noise=0.0_seed=2.csv
Saved log_d data for Measured (n=50, noise=0.0, seed=3) to 3D-Data\Measured_D\rss_n=50_noise=0.0_seed=3.csv
Saved log_d data for Measured (n=50, noise=0.0, seed=4) to 3D-Data\Measured_D\rss_n=50_noise=0.0_seed=4.csv
Saved log_d data for Measured (n=50, noise=0.0, seed=5) to 3D-Data\Measured_D\rss_n=50_noise=0.0_seed=5.csv
Saved log_d data for Measured (n=50, noise=0.0, seed=6) to 3D-Data\Measured_D\rss_n=50_noise=0.0_seed=6.csv
Saved log_d data for Measured (n=50, noise=0.0, seed=7) to 3D-Data\Measured_D\rss_n=50_noise=0.0_seed=7.csv
Saved log_d data for Measured (n=50, noise=0.0, seed=8) to 3D-Data\Measured_D\rss_n=50_noise=0.0_seed=8.csv
Saved log_d data for Measure

In [4]:
ns= np.array([50])
categories=['Measured']
noise_levels= [0.,0.000006, 0.000019,0.00006,0.00019]
base_dir='3D-Data'
# test data creation on dense grid
for category in categories:
    for noise_level in noise_levels:
        for n in ns:
            my_scene=Scene()
            use_measured_data= (category =='Measured')
            my_scene.generate_sven_lights(use_measured_data=use_measured_data)
            my_scene.generate_diode_volume(n=n,volume_boundary=[[-4,4],[-4,4],[0,h]])
            data= my_scene.generate_rss_table_with_diode_coords_and_distances(noise_std=noise_level,seed=0)
            data= data.drop(columns=["X","Y","Z"])
            category_dir= os.path.join(base_dir,"Measured_D")
            os.makedirs(category_dir,exist_ok=True)
            file_name= f"gnd_n={n}_noise={noise_level}.csv"
            file_path = os.path.join(category_dir, file_name)
            data.to_csv(file_path, index=False)
            print(f"Saved data for {category} (n={n}, noise={noise_level}) to {file_path}")

Saved data for Measured (n=50, noise=0.0) to 3D-Data\Measured_D\gnd_n=50_noise=0.0.csv
Saved data for Measured (n=50, noise=6e-06) to 3D-Data\Measured_D\gnd_n=50_noise=6e-06.csv
Saved data for Measured (n=50, noise=1.9e-05) to 3D-Data\Measured_D\gnd_n=50_noise=1.9e-05.csv
Saved data for Measured (n=50, noise=6e-05) to 3D-Data\Measured_D\gnd_n=50_noise=6e-05.csv
Saved data for Measured (n=50, noise=0.00019) to 3D-Data\Measured_D\gnd_n=50_noise=0.00019.csv


In [7]:
# Define paths and parameters
base_dir="3D-Data"
category = "Measured_D"
n= 50
noise_level= 0.
scaler_filename = "scaler_log_d.pkl"


category_dir= os.path.join(base_dir,category)

file_name= f"gnd_n={n}_noise={noise_level}.csv"
file_path = os.path.join(category_dir, file_name)

# Load data
data = pd.read_csv(file_path)
columns = data.columns  # Store original column names for later

# Preprocess data: Apply logarithm before scaling
regenerate_scaler = True
data_log = np.log(data.values)  # Add a small constant to avoid log(0)
print(data_log)
# Scaling
if regenerate_scaler:
    scaler = StandardScaler()
    scaler.fit(data_log)
    with open(scaler_filename, "wb") as f:
        pickle.dump(scaler, f)
    print(f"Scaler saved as '{scaler_filename}'")
else:
    with open(scaler_filename, "rb") as f:
        scaler = pickle.load(f)


# Transform the data
scaled_data = pd.DataFrame(scaler.transform(data_log), columns=columns)
print(scaled_data)

# Save scaled data if needed
save_dir = os.path.join(base_dir,"Measured_log_d_Normalized")
scaled_file_path = os.path.join(save_dir, f"gnd_n={n}_noise={noise_level}.csv")
scaled_data.to_csv(scaled_file_path, index=False)
print(f"Scaled data saved as '{scaled_file_path}'")


[[ -5.88765787  -4.82739808  -6.11305778 ...   2.08719363   2.08719363
    2.28735549]
 [ -5.89080294  -4.8122566   -6.11988997 ...   2.0817084    2.0817084
    2.28368647]
 [ -5.89445746  -4.79716728  -6.12711141 ...   2.07624255   2.07624255
    2.28004371]
 ...
 [ -7.96639844 -10.93840249  -8.79374675 ...   1.87708751   1.87708751
    1.18507419]
 [ -8.15221719 -11.19171709  -9.03095772 ...   1.87439176   1.87439176
    1.17422777]
 [ -8.36283405 -11.5065361   -9.28858037 ...   1.87180218   1.87180218
    1.16363885]]
Scaler saved as 'scaler_log_d.pkl'
            RSS0      RSS1      RSS2      RSS3      RSS4        D0        D1  \
0      -1.326273  0.436884 -0.655324 -0.499052 -1.297849  2.026841  0.384126   
1      -1.330257  0.448659 -0.661019 -0.503888 -1.310967  2.002228  0.349999   
2      -1.334887  0.460393 -0.667039 -0.509162 -1.324856  1.977665  0.315621   
3      -1.340073  0.472080 -0.673442 -0.514771 -1.339447  1.953155  0.280992   
4      -1.345910  0.483716 -0.680567 -

In [8]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Define paths
base_dir = "3D-Data"
category = "Measured_D"
normalized_category = "Measured_log_d_Normalized"
scaler_filename = "scaler_log_d.pkl"

# Paths for original and normalized data
category_dir = os.path.join(base_dir, category)
normalized_category_dir = os.path.join(base_dir, normalized_category)

# Ensure the normalized folder exists
os.makedirs(normalized_category_dir, exist_ok=True)

# Load the precomputed scaler
with open(scaler_filename, "rb") as f:
    loaded_scaler = pickle.load(f)
print(f"Loaded scaler from '{scaler_filename}'")

# Iterate over all CSV files in the Measured_D folder
for file_name in os.listdir(category_dir):
    if file_name.endswith(".csv"):  # Process all CSV files
        file_path = os.path.join(category_dir, file_name)
        data = pd.read_csv(file_path)

        # Apply log transformation (add small constant to avoid log(0))
        data_log = np.log(np.abs(data.values))  # 1e-8 prevents issues with log(0)

        # Normalize the data using the precomputed scaler
        normalized_data = pd.DataFrame(loaded_scaler.transform(data_log), columns=data.columns)

        # Save the normalized data to the separate Measured_log_d_Normalized folder
        normalized_file_path = os.path.join(normalized_category_dir, file_name)
        normalized_data.to_csv(normalized_file_path, index=False)

        print(f"Saved log_d normalized data to {normalized_file_path}")


Loaded scaler from 'scaler_log_d.pkl'
Saved log_d normalized data to 3D-Data\Measured_log_d_Normalized\gnd_n=50_noise=0.0.csv
Saved log_d normalized data to 3D-Data\Measured_log_d_Normalized\gnd_n=50_noise=0.00019.csv
Saved log_d normalized data to 3D-Data\Measured_log_d_Normalized\gnd_n=50_noise=1.9e-05.csv
Saved log_d normalized data to 3D-Data\Measured_log_d_Normalized\gnd_n=50_noise=6e-05.csv
Saved log_d normalized data to 3D-Data\Measured_log_d_Normalized\gnd_n=50_noise=6e-06.csv
Saved log_d normalized data to 3D-Data\Measured_log_d_Normalized\rss_n=1000_noise=0.0_seed=0.csv
Saved log_d normalized data to 3D-Data\Measured_log_d_Normalized\rss_n=1000_noise=0.0_seed=1.csv
Saved log_d normalized data to 3D-Data\Measured_log_d_Normalized\rss_n=1000_noise=0.0_seed=2.csv
Saved log_d normalized data to 3D-Data\Measured_log_d_Normalized\rss_n=1000_noise=0.0_seed=3.csv
Saved log_d normalized data to 3D-Data\Measured_log_d_Normalized\rss_n=1000_noise=0.0_seed=4.csv
Saved log_d normalized da

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Define paths
base_dir = "3D-Data"
category = "Measured_relative"
normalized_category = "Measured_relative_Normalized"
scaler_filename = "scaler_relative.pkl"

# Paths for original and normalized data
category_dir = os.path.join(base_dir, category)
normalized_category_dir = os.path.join(base_dir, normalized_category)

# Ensure the normalized folder exists
os.makedirs(normalized_category_dir, exist_ok=True)

# Load the precomputed scaler
with open(scaler_filename, "rb") as f:
    loaded_scaler = pickle.load(f)
print(f"Loaded scaler from '{scaler_filename}'")

# Iterate over all CSV files in the Measured_Relative_Test folder
for file_name in os.listdir(category_dir):
    if file_name.endswith(".csv"):  # Process all CSV files
        file_path = os.path.join(category_dir, file_name)
        data = pd.read_csv(file_path)

        # Separate RSS ratio columns and coordinate columns
        rss_ratio_columns = [col for col in data.columns if "/" in col]  # Detect RSS ratio columns
        coordinate_columns = ["X", "Y", "Z"]

        # Apply log transformation only to RSS ratio columns
        data_log = data.copy()
        data_log[rss_ratio_columns] = np.log(np.abs(data[rss_ratio_columns]))  # Log transform only RSS ratio columns

        # Normalize the data using the precomputed scaler
        normalized_data = pd.DataFrame(loaded_scaler.transform(data_log.values), columns=data.columns)

        # Save the normalized data to the separate Measured_Relative_Normalized folder
        normalized_file_path = os.path.join(normalized_category_dir, file_name)
        normalized_data.to_csv(normalized_file_path, index=False)

        print(f"Saved relative normalized data to {normalized_file_path}")


Loaded scaler from 'scaler_relative.pkl'
Saved relative normalized data to 3D-Data\Measured_relative_Normalized\relative_gnd_n=50_noise=0.0.csv
Saved relative normalized data to 3D-Data\Measured_relative_Normalized\relative_gnd_n=50_noise=0.00019.csv
Saved relative normalized data to 3D-Data\Measured_relative_Normalized\relative_gnd_n=50_noise=1.9e-05.csv
Saved relative normalized data to 3D-Data\Measured_relative_Normalized\relative_gnd_n=50_noise=6e-05.csv
Saved relative normalized data to 3D-Data\Measured_relative_Normalized\relative_gnd_n=50_noise=6e-06.csv
Saved relative normalized data to 3D-Data\Measured_relative_Normalized\relative_rss_n=1000_noise=0.0_seed=0.csv
Saved relative normalized data to 3D-Data\Measured_relative_Normalized\relative_rss_n=1000_noise=0.0_seed=1.csv
Saved relative normalized data to 3D-Data\Measured_relative_Normalized\relative_rss_n=1000_noise=0.0_seed=2.csv
Saved relative normalized data to 3D-Data\Measured_relative_Normalized\relative_rss_n=1000_noise